In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             precision_recall_curve)
import xgboost as xgb
import optuna
from optuna.samplers import TPESampler
import shap
import joblib
from imblearn.under_sampling import NearMiss
import warnings

warnings.filterwarnings('ignore')

# ============================================================================
# 1. Data Loading
# ============================================================================
print("=" * 80)
print("1. Data Loading")
print("=" * 80)
df = pd.read_csv('selected_data_for_modeling.csv')

X = df.drop(['ID', 'PERF_12M'], axis=1)
y = df['PERF_12M']

print(f"Full dataset shape: {X.shape}")
print(f"Total bankruptcy cases: {y.sum()} ({(y.sum()/len(y))*100:.2f}%)")

# ============================================================================
# 2. NearMiss Undersampling (as per paper methodology)
# ============================================================================
print("\n" + "=" * 80)
print("2. Applying NearMiss Undersampling (1:1 ratio)")
print("=" * 80)

# Paper rationale: Applied 1:1 ratio to address class imbalance
nm = NearMiss(version=1, n_neighbors=3)
X_resampled, y_resampled = nm.fit_resample(X, y)

print(f"Dataset shape after resampling: {X_resampled.shape}")
print(f"Bankruptcy cases after resampling: {y_resampled.sum()}")
print(f"Solvent cases after resampling: {(y_resampled==0).sum()}")
print("Class ratio adjusted to 1:1.")

# ----------------------------------------------------------------------------
# [Added] Save resampled data (used in Step 8 visualisation and Agent steps)
# ----------------------------------------------------------------------------
df_resampled = pd.DataFrame(X_resampled, columns=X.columns)
df_resampled['PERF_12M'] = y_resampled

df_resampled.to_csv('resampled_data_final.csv', index=False, encoding='utf-8-sig')
print(">> 'resampled_data_final.csv' saved (for visualisation)")
# ----------------------------------------------------------------------------

# Train/Test Split (80:20)
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled
)

# scale_pos_weight set to 1.0 since ratio is 1:1
pos_weight = 1.0

# ============================================================================
# Utility Functions
# ============================================================================
def find_optimal_threshold(y_true, y_proba):
    precision, recall, thresholds = precision_recall_curve(y_true, y_proba)
    # Guard against division by zero
    with np.errstate(divide='ignore', invalid='ignore'):
        f1_scores = 2 * (precision * recall) / (precision + recall)
    f1_scores = np.nan_to_num(f1_scores)

    if len(f1_scores) == 0:
        return 0.5, 0.0

    optimal_idx = np.argmax(f1_scores)
    if optimal_idx < len(thresholds):
        optimal_threshold = thresholds[optimal_idx]
    else:
        optimal_threshold = 0.5
    return optimal_threshold, f1_scores[optimal_idx]

def evaluate_model(model, X_test, y_test, model_name):
    y_prob = model.predict_proba(X_test)[:, 1]

    # Find optimal threshold on test set (for performance verification)
    threshold, f1 = find_optimal_threshold(y_test, y_prob)
    y_pred = (y_prob >= threshold).astype(int)

    print(f"\n[{model_name}] Results")
    print(f"AUC: {roc_auc_score(y_test, y_prob):.4f}")
    print(f"Optimal Threshold: {threshold:.4f}")
    print(f"F1 Score: {f1:.4f}")

    cm = confusion_matrix(y_test, y_pred)
    print(f"Confusion Matrix:\n{cm}")
    print(f"Correctly predicted bankruptcy (TP): {cm[1,1]} / {cm[1,0]+cm[1,1]}")

    return {'Model': model_name, 'Threshold': threshold, 'AUC': roc_auc_score(y_test, y_prob)}

# ============================================================================
# 3. Base Model Training (Optuna)
# ============================================================================
print("\n" + "=" * 80)
print("3. XGBoost Training with Optuna Hyperparameter Optimisation")
print("=" * 80)

def objective(trial):
    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'tree_method': 'hist',
        'random_state': 42,
        'n_jobs': -1,
        'scale_pos_weight': pos_weight,  # 1.0

        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 7)
    }

    # Increased K-Fold splits for stability given small dataset (~400 samples)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []

    for tr_idx, val_idx in cv.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

        model = xgb.XGBClassifier(**params)
        model.fit(X_tr, y_tr, verbose=False)
        scores.append(roc_auc_score(y_val, model.predict_proba(X_val)[:, 1]))

    return np.mean(scores)

study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=30)  # Completes quickly given small dataset

best_params = study.best_params
best_params.update({'random_state': 42, 'n_jobs': -1})

print(f"Best CV AUC: {study.best_value:.4f}")

# Train final model
base_model = xgb.XGBClassifier(**best_params)
base_model.fit(X_train, y_train)

# Evaluate
res = evaluate_model(base_model, X_test, y_test, 'Final Model')

# ============================================================================
# 4. Save Outputs (required for Agent steps)
# ============================================================================
X_test.to_csv('X_test_final.csv', index=False)
y_test.to_csv('y_test_final.csv', index=False)

joblib.dump(base_model, 'base_model_final.pkl')
joblib.dump(res['Threshold'], 'base_model_threshold_final.pkl')
joblib.dump(list(X.columns), 'selected_features_final.pkl')  # Save feature names

print("\nAll files saved. Proceed to the next step (DiCE).")

1. Data Loading
Full dataset shape: (15482, 64)
Total bankruptcy cases: 266 (1.72%)

2. Applying NearMiss Undersampling (1:1 ratio)


[I 2026-03-20 00:13:22,024] A new study created in memory with name: no-name-6ccd10d6-9a4e-4a26-996d-4be2a5fd64f1


Dataset shape after resampling: (532, 64)
Bankruptcy cases after resampling: 266
Solvent cases after resampling: 266
Class ratio adjusted to 1:1.
>> 'resampled_data_final.csv' saved (for visualisation)

3. XGBoost Training with Optuna Hyperparameter Optimisation


[I 2026-03-20 00:13:22,790] Trial 0 finished with value: 0.9558139534883721 and parameters: {'max_depth': 5, 'learning_rate': 0.28570714885887566, 'n_estimators': 380, 'min_child_weight': 5}. Best is trial 0 with value: 0.9558139534883721.
[I 2026-03-20 00:13:23,044] Trial 1 finished with value: 0.952048726467331 and parameters: {'max_depth': 3, 'learning_rate': 0.055238410897498764, 'n_estimators': 76, 'min_child_weight': 7}. Best is trial 0 with value: 0.9558139534883721.
[I 2026-03-20 00:13:23,263] Trial 2 finished with value: 0.958361018826135 and parameters: {'max_depth': 6, 'learning_rate': 0.21534104756085318, 'n_estimators': 59, 'min_child_weight': 7}. Best is trial 2 with value: 0.958361018826135.
[I 2026-03-20 00:13:23,700] Trial 3 finished with value: 0.9534883720930232 and parameters: {'max_depth': 7, 'learning_rate': 0.07157834209670008, 'n_estimators': 132, 'min_child_weight': 2}. Best is trial 2 with value: 0.958361018826135.
[I 2026-03-20 00:13:24,219] Trial 4 finished 

Best CV AUC: 0.9602

[Final Model] Results
AUC: 0.9801
Optimal Threshold: 0.6989
F1 Score: 0.9608
Confusion Matrix:
[[54  0]
 [ 4 49]]
Correctly predicted bankruptcy (TP): 49 / 53

All files saved. Proceed to the next step (DiCE).
